# Collect all data signals

Fetch all data signals since 2025-01-01. 
Combine them into one or multiple tables to share with Will.
Investments
All companies established or added to the database this year
(double check our criteria for Slack updates)
All new funding rounds from this year
Research projects
All research projects added or starting this year
(check how much signal we get if only use those that start this year)


We could also monitor new outputs of already running projects in future iterations
Policy
All debates and highlighted quotes from this year

In [3]:
from discovery_utils.getters import crunchbase
from discovery_utils import PROJECT_DIR, LOCAL_VECTOR_DB_PATH, logging

from typing import Literal

In [5]:
from discovery_utils.utils.llm.batch_check import LLMProcessor, generate_relevance_check_system_message

In [ ]:
CB = crunchbase.CrunchbaseGetter()


In [ ]:
import importlib
importlib.reload(crunchbase);

In [ ]:
CB_ = crunchbase.CrunchbaseGetter()
CB_._organisations_enriched = CB._organisations_enriched
CB_._funding_rounds_enriched = CB._funding_rounds_enriched

In [ ]:
from datetime import date
from dateutil.relativedelta import relativedelta

def define_dates(year: int, quarter: int, offset_start_date: int = 0) -> tuple[str, str]:
    """Get start and end dates for a given quarter of a year."""
    if quarter not in {1, 2, 3, 4}:
        raise ValueError("Quarter must be between 1 and 4.")

    # Determine the start month of the quarter
    start_month = (quarter - 1) * 3 + 1
    start_date = date(year, start_month, 1) - relativedelta(days=offset_start_date)
    
    # End date is the last day of the quarter
    end_date = start_date + relativedelta(months=3, days=-1)

    return start_date.isoformat(), end_date.isoformat()

In [ ]:
mission = "ASF"
year = 2025
quarter = 1

start_date, end_date = define_dates(year=year, quarter=quarter, offset_start_date=1)

In [ ]:
orgs = CB_.get_companies_in_nesta_categories("mission_labels", ["ASF"])

In [ ]:
CB.funding_rounds_enriched.investment_type.unique()

In [ ]:
countries = (
    crunchbase.REGION_TO_COUNTRIES["UK"]
    + crunchbase.REGION_TO_COUNTRIES["Europe"]
    + crunchbase.REGION_TO_COUNTRIES["North America + Australia"]
)

In [ ]:
cols_funding_rounds = [
    "funding_round_name", 
    "org_name",
    'mission_labels',
    'topic_labels',    
    "cb_url",
    "country_code", 
    "region_nesta",
    "region",
    "city",
    "year",
    "announced_on",
    "investment_type", 
    "investment_stage",
    "raised_amount_gbp",
    "raised_amount_usd",
    "raised_amount", 
    "raised_amount_currency_code",
    "post_money_valuation_usd", 
    "post_money_valuation",
    "post_money_valuation_currency_code", 
    "investor_count",
    "investor_name",
    "is_lead_investor",
    "investor_types",
]

In [ ]:
cols_companies = [
    "name", 
    "short_description", 
    "founded_on", 
    'created_at',    
    "cb_url", 
    "homepage_url", 
    'mission_labels',
    'topic_labels',
    "rank", 
    "country_code", 
    "region_nesta",
    "region", 
    "city", 
    "status", 
    "category_list", 
    "closed_on", 
    "employee_count", 
    "email", 
    "phone", 
    "facebook_url", 
    "linkedin_url", 
    "twitter_url", 
    "logo_url", 
    "num_exits", 
    "num_funding_rounds", 
    "last_funding_on", 
    "investment_funding_gbp", 
    "num_investment_rounds", 
    "grant_funding_gbp", 
    "num_grants", 
    "total_funding_gbp", 
    "smart_money", 
]

In [ ]:
rounds = (
    CB.funding_rounds_enriched
    .query("announced_on >= @start_date and announced_on <= @end_date")
    .query("country_code in @countries")
    .query("org_id in @orgs.id.to_list()")
    .assign(investment_category = lambda df: df.investment_type.map(crunchbase.investment_type_to_stage()))
)

In [ ]:
def agg_list(x) -> str:
    return ", ".join([str(item) for item in list(x)])

rounds_investors = (
    rounds[['funding_round_id', 'investor_name', 'is_lead_investor', 'investor_types', 'investor_url']]
    .groupby('funding_round_id').agg(agg_list).reset_index()
)

_rounds = (
    rounds
    .drop(columns=['investor_name', 'is_lead_investor', 'investor_types', 'investor_url'], axis=1)
    .drop_duplicates("funding_round_id")
    .merge(rounds_investors, on='funding_round_id')
    .assign(investment_stage = lambda df: df.investment_type.map(crunchbase.investment_type_to_stage()))
    .merge(orgs[['id', 'mission_labels', 'topic_labels']].rename(columns={'id': 'org_id'}), on='org_id', how='left')
    .assign(region_nesta = lambda df: df.country_code.map(crunchbase.country_to_region()))
    .fillna("")
    .astype(str)
    .reset_index(drop=True)
)[cols_funding_rounds]


In [ ]:
_rounds

In [ ]:
new_orgs = (
    orgs
    .query("(founded_on >= @start_date and founded_on <= @end_date) or (created_at >= @start_date and created_at <= @end_date)")
    .query("country_code in @countries")
    .assign(region_nesta = lambda df: df.country_code.map(crunchbase.country_to_region()))
    .sort_values("founded_on", ascending=False)
    .fillna("")
    .astype(str)
    .reset_index(drop=True)    
)[cols_companies]

In [ ]:
from discovery_utils.utils import google
import os
import importlib
importlib.reload(os);
importlib.reload(google);

In [ ]:
sheet_id = "1w2nSas1LwmPQY9HK-drrxIdGpDV3wU3pePDibHPVqL8"

In [ ]:
google.upload_data_to_gsheet(sheet_id, {"crunchbase_funding": _rounds})
google.format_gsheet(sheet_id, "crunchbase_funding", freeze_cols=2)

In [ ]:


google.upload_data_to_gsheet(sheet_id, {"crunchbase_companies": new_orgs})
google.format_gsheet(sheet_id, "crunchbase_companies", freeze_cols=4)

In [ ]:
google.format_gsheet(sheet_id, "crunchbase_funding", freeze_cols=2)

## Gateway to Research

In [ ]:
from datetime import datetime

def convert_to_date(x: str) -> str:
    try:
        return datetime.fromtimestamp(x / 1000).strftime('%Y-%m-%d')
    except:
        return ""

In [ ]:
cols_projects = [
    'title',
    'is_relevant',       
    'mission_labels',
    'topic_labels',
    'status', 
    'grantCategory',
    'leadFunder',
    'abstractText',
    'techAbstractText',
    'potentialImpact',
    'start',
    'end',
    'amount',
    'url',
]

In [ ]:
from discovery_utils.getters import gtr
from discovery_utils.utils import keywords as kw
import pandas as pd

In [ ]:
GTR = gtr.GtrGetter()

In [ ]:
new_projects = (
    GTR.projects_enriched
    .assign(created = lambda df: df.created.apply(convert_to_date))
    .query("(start >= @start_date and start <= @end_date)")
)
new_projects_text = GTR.get_projects_text().query("id in @new_projects.id.to_list()")

In [ ]:
if len(new_projects) > 0:
    enrichment_df = kw.enrich_topic_labels(new_projects_text)

In [ ]:
filename = f"llm_check.jsonl"
config_filename = f"config_{mission}.yaml"
system_message = generate_system_message(config_filename)
fields = [
    {"name": "is_relevant", "type": "str", "description": "A one-word answer: 'yes' or 'no'."},
]

check_data = dict(zip(new_projects_text['id'], new_projects_text['text']))

In [ ]:
processor = LLMProcessor(
    output_path=filename,
    system_message=system_message,
    session_name="mission_radar",
    output_fields=fields,
)

processor.run(check_data, batch_size=15, sleep_time=0.5)

In [ ]:
relevant_check_df = pd.read_json(filename, lines=True)

In [ ]:
_new_projects = (
    new_projects
    .merge(enrichment_df, on='id', how='left')
    .assign(mission_labels = lambda df: df.mission_labels.apply(lambda x: x.split(",") if (type(x) is str) else []))
    .explode('mission_labels')
    .query("mission_labels in @mission")
    .merge(relevant_check_df, left_on='id', right_on='id', how='left')
    .sort_values(["is_relevant", "start"], ascending=False)
    .drop_duplicates("id")
    .fillna("")
    .astype(str)
    .reset_index(drop=True)    
)[cols_projects]

In [ ]:
_new_projects

In [ ]:
google.upload_data_to_gsheet(sheet_id, {"ukri_projects": _new_projects})

In [ ]:
google.format_gsheet(sheet_id, "ukri_projects", freeze_cols=4)